# Week 55, Day 5: Randomised algorithms

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w055d5.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
import random

a = random.Random(42)
b = random.Random(42)
print([a.randint(1, 6) for _ in range(5)])
print([b.randint(1, 6) for _ in range(5)])
c = random.Random(7)
print([c.randint(1, 6) for _ in range(5)])

In [ ]:
import random
from collections import Counter

def fisher_yates(items, rng):
    for i in range(len(items) - 1, 0, -1):
        j = rng.randint(0, i)
        items[i], items[j] = items[j], items[i]

def naive(items, rng):
    for i in range(len(items)):
        j = rng.randrange(len(items))
        items[i], items[j] = items[j], items[i]

rng = random.Random(1)
for shuffle in (fisher_yates, naive):
    counts = Counter()
    for _ in range(60_000):
        a = list("abc")
        shuffle(a, rng)
        counts["".join(a)] += 1
    print(shuffle.__name__, sorted(counts.values()))

In [ ]:
import random

def quickselect(items, k, rng):
    """The k-th smallest (from 0), without sorting."""
    items = list(items)
    while True:
        pivot = items[rng.randrange(len(items))]
        lower = [x for x in items if x < pivot]
        equal = sum(1 for x in items if x == pivot)
        if k < len(lower):
            items = lower
        elif k < len(lower) + equal:
            return pivot
        else:
            k -= len(lower) + equal
            items = [x for x in items if x > pivot]

rng = random.Random(3)
data = [rng.randint(0, 10 ** 6) for _ in range(100_001)]
print(quickselect(data, 50_000, rng) == sorted(data)[50_000])

In [ ]:
import random

def is_probable_prime(n, rng, rounds=20):
    if n < 2:
        return False
    for p in (2, 3, 5, 7, 11, 13):
        if n % p == 0:
            return n == p
    s, d = 0, n - 1
    while d % 2 == 0:
        s, d = s + 1, d // 2
    for _ in range(rounds):
        a = rng.randrange(2, n - 1)
        x = pow(a, d, n)
        if x in (1, n - 1):
            continue
        for _ in range(s - 1):
            x = x * x % n
            if x == n - 1:
                break
        else:
            return False
    return True

rng = random.Random(5)
print(is_probable_prime(561, rng), pow(2, 560, 561), is_probable_prime(2 ** 127 - 1, rng), is_probable_prime(2 ** 128 + 1, rng))

> **Key idea.** Pass a seeded random.Random into randomised code so it's repeatable and testable; use secrets for anything secure. Las Vegas algorithms are always right with random running time; Monte Carlo ones have fixed time and a small, repeatable-away chance of error. Fisher-Yates swaps each position i with a random one from 0 to i for a fair shuffle. Quickselect with a random pivot finds the k-th smallest in expected O(n). Reservoir sampling keeps k uniform items from a stream of unknown length. Miller-Rabin tests primality with random bases, beating Carmichael numbers, and Freivalds' check verifies a matrix product in O(n²) per round.

### Exercise: A fair shuffle

Write `shuffle(items, rng)`: shuffle the list **in place** with Fisher-Yates, using only `rng.randint` (no `rng.shuffle`). Then `random_permutation(n, rng)`: a new shuffled list of 0 to n − 1. The checks count how often each order of 4 items appears.

In [ ]:
def shuffle(items, rng):
    ___

def random_permutation(n, rng):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random\nfrom collections import Counter\ndef test_shuffle():\n    \"Fisher-Yates\"\n    rng = random.Random(1)\n    a = list(range(10))\n    shuffle(a, rng)\n    assert sorted(a) == list(range(10)) and shuffle([], rng) is None\n    counts = Counter()\n    for _ in range(48_000):\n        b = list(\"abcd\")\n        shuffle(b, rng)\n        counts[\"\".join(b)] += 1\n    assert len(counts) == 24 and min(counts.values()) > 1700 and max(counts.values()) < 2300, counts\n    assert sorted(random_permutation(50, random.Random(2))) == list(range(50))\n    assert random_permutation(8, random.Random(3)) == random_permutation(8, random.Random(3))\n    assert \".shuffle(\" not in source and \"sample(\" not in source")

### Exercise: Quickselect

Write `kth_smallest(items, k, rng)` (k from 0) with quickselect and a random pivot, iteratively, raising `IndexError` for k out of range. Then `median(items, rng)`: the middle item for an odd count, or the average of the two middle items for an even count (`ValueError` if empty). Don't sort.

In [ ]:
def kth_smallest(items, k, rng):
    ___

def median(items, rng):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random, time\ndef test_select():\n    \"quickselect\"\n    rng = random.Random(1)\n    assert kth_smallest([5, 1, 4, 2, 3], 0, rng) == 1 and kth_smallest([5, 1, 4, 2, 3], 4, rng) == 5\n    assert kth_smallest([7, 7, 7], 1, rng) == 7 and median([3, 1, 2], rng) == 2 and median([4, 1, 3, 2], rng) == 2.5\n    for bad, exc in ((lambda: kth_smallest([1], 1, rng), IndexError), (lambda: median([], rng), ValueError)):\n        try:\n            bad()\n            assert False\n        except exc:\n            pass\n    for _ in range(200):\n        data = [rng.randint(0, 20) for _ in range(rng.randint(1, 30))]\n        k = rng.randrange(len(data))\n        assert kth_smallest(data, k, rng) == sorted(data)[k]\n    big = list(range(200_000))\n    t0 = time.perf_counter()\n    assert kth_smallest(big, 123_456, rng) == 123_456 and median(big[::-1], rng) == 99_999.5\n    assert time.perf_counter() - t0 < 6\n    assert \"sorted(\" not in source and \".sort(\" not in source")

### Exercise: A sample of a stream

Write `reservoir(stream, k, rng)`: k items chosen uniformly from an iterable you may go through only **once** (fewer if the stream is shorter), with Algorithm R. The checks feed a stream that can't be measured with `len` and count how often each item is chosen.

In [ ]:
def reservoir(stream, k, rng):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random\nfrom collections import Counter\nclass Stream:\n    def __init__(self, n):\n        self.n = n\n    def __iter__(self):\n        return iter(range(self.n))\ndef test_reservoir():\n    \"Algorithm R\"\n    rng = random.Random(1)\n    assert sorted(reservoir(Stream(3), 5, rng)) == [0, 1, 2] and reservoir(Stream(0), 2, rng) == []\n    counts = Counter()\n    for _ in range(20_000):\n        counts.update(reservoir(Stream(10), 3, rng))\n    assert all(5600 < counts[i] < 6400 for i in range(10)), counts\n    s = reservoir(Stream(100_000), 10, rng)\n    assert len(s) == 10 and len(set(s)) == 10")

### Exercise: Miller-Rabin

Write `is_prime(n, rng=None, rounds=20)`: Miller-Rabin. With `rng=None`, use the fixed bases 2, 3, 5, 7, 11, 13, 17, 19, 23, 29, 31 and 37, which make the answer **certain** for n below 3.3 × 10²⁴; otherwise use `rounds` random bases. Then `next_prime(n)`: the smallest prime greater than n, with the fixed bases.

In [ ]:
def is_prime(n, rng=None, rounds=20):
    ___

def next_prime(n):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random\ndef test_miller():\n    \"primality\"\n    limit = 20_000\n    sieve = [True] * (limit + 1)\n    sieve[0] = sieve[1] = False\n    for p in range(2, int(limit ** 0.5) + 1):\n        if sieve[p]:\n            for m in range(p * p, limit + 1, p):\n                sieve[m] = False\n    assert [n for n in range(limit + 1) if is_prime(n)] == [n for n in range(limit + 1) if sieve[n]]\n    for carmichael in (561, 1105, 1729, 2465, 2821, 6601, 8911, 41041, 825265):\n        assert not is_prime(carmichael) and not is_prime(carmichael, random.Random(1))\n    assert is_prime(2 ** 61 - 1) and is_prime(10 ** 9 + 7) and not is_prime((2 ** 31 - 1) * (2 ** 61 - 1))\n    assert is_prime(2 ** 127 - 1, random.Random(2)) and not is_prime(2 ** 128 + 1, random.Random(2))\n    assert next_prime(10 ** 12) == 1000000000039 and next_prime(1) == 2 and next_prime(13) == 17")

### Exercise: Checking a product

Write `multiply(a, b)` for square matrices (lists of lists), and `probably_equal(a, b, c, rng, rounds=20)`: Freivalds' check that a × b == c, comparing a(b r) with c r for random 0/1 vectors r, O(n²) per round.

In [ ]:
def multiply(a, b):
    ___

def probably_equal(a, b, c, rng, rounds=20):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random, time\ndef test_freivalds():\n    \"random checks\"\n    rng = random.Random(1)\n    a = [[1, 2], [3, 4]]\n    b = [[5, 6], [7, 8]]\n    assert multiply(a, b) == [[19, 22], [43, 50]]\n    assert probably_equal(a, b, [[19, 22], [43, 50]], rng) and not probably_equal(a, b, [[19, 22], [43, 51]], rng)\n    for _ in range(50):\n        n = rng.randint(1, 6)\n        a = [[rng.randint(-5, 5) for _ in range(n)] for _ in range(n)]\n        b = [[rng.randint(-5, 5) for _ in range(n)] for _ in range(n)]\n        c = multiply(a, b)\n        assert probably_equal(a, b, c, rng)\n        c[rng.randrange(n)][rng.randrange(n)] += rng.choice([-1, 1])\n        assert not probably_equal(a, b, c, rng)\n    n = 150\n    a = [[rng.randint(0, 9) for _ in range(n)] for _ in range(n)]\n    b = [[rng.randint(0, 9) for _ in range(n)] for _ in range(n)]\n    c = multiply(a, b)\n    c[37][91] += 1\n    t0 = time.perf_counter()\n    assert not probably_equal(a, b, c, rng)\n    assert time.perf_counter() - t0 < 2")

### Bug hunt: Nobody stays put

Fisher-Yates picks j from 0 to i **inclusive**. With `randrange(i)` (0 to i − 1), no item can ever stay in its place, so of the six orders of three items only the two "rotations" appear and this prints `2`. Include i so it prints `6`.

In [ ]:
import random

rng = random.Random(1)
seen = set()
for _ in range(1000):
    a = [0, 1, 2]
    for i in range(len(a) - 1, 0, -1):
        j = rng.randrange(i)
        a[i], a[j] = a[j], a[i]
    seen.add(tuple(a))
print(len(seen))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"every order\"\n    expect_output(\"6\")")

### Bug hunt: Fooled by Carmichael

This is Fermat's test, `pow(a, n - 1, n) == 1` for a few bases, and 561 = 3 × 11 × 17 fools it for every base coprime to 561, so it prints `True`. Add Miller-Rabin's square-root check (write n − 1 = 2ˢ × d, look for n − 1 among the squarings) so it prints `False`.

In [ ]:
def looks_prime(n):
    return all(pow(a, n - 1, n) == 1 for a in (2, 5, 7))

print(looks_prime(561))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"Miller-Rabin\"\n    expect_output(\"False\")")

### Bug hunt: A quarter of a circle

Random points in the unit square land inside the quarter circle x² + y² ≤ 1 with probability π/4, so π is about **4** times the fraction inside. This multiplies by 2 and prints `1.57`. Fix it so it prints `3.14`.

In [ ]:
import random

rng = random.Random(1)
inside = sum(1 for _ in range(200_000) if rng.random() ** 2 + rng.random() ** 2 <= 1)
print(round(2 * inside / 200_000, 2))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"four times\"\n    expect_output(\"3.14\")")

### Puzzle: Reservoir sampling

Arrange the lines of Algorithm R. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them



In [ ]:
p = ___   # True or False: random.Random(9).random() == random.Random(9).random()
q = ___   # "Las Vegas" or "Monte Carlo": always correct, random running time
r = ___   # the chance an item ends up in a reservoir of size 2 from a stream of 10, as a fraction like 0.5
s = ___   # n − 1 = 2^s × d for n = 97: the value of s
t = ___   # True or False: Fermat's test can call a Carmichael number prime

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert p is True and q == \"Las Vegas\" and r == 0.2, \"p, q and r\"\n    assert s == 5 and t is True, \"s and t\"")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: a Bloom filter

A **Bloom filter** (Burton Bloom, 1970) answers "have I seen this?" in a tiny, fixed amount of memory, at the price of occasional **false positives** (never false negatives). It's an array of m bits and k hash functions: adding an item sets the k bits its hashes point to; an item is "probably present" only if all k of its bits are set. Browsers and databases use them to skip expensive lookups for things that are certainly absent.

Write `BloomFilter(m, k)` with the bits in one integer (Monday's bit operations):

- `positions(item)`: k positions from 0 to m − 1 for a string, by **double hashing**: take two numbers h1 and h2 from the SHA-256 digest of the item (`hashlib.sha256(item.encode()).digest()`; the first 8 bytes and the next 8 as big-endian integers), then position i is `(h1 + i * h2) % m`. (Python's built-in `hash` of a string changes between runs, so it's no good here.)
- `add(item)` and `item in bloom` (`__contains__`); `len(bloom)` is the number of items added.
- `expected_false_rate()`: (1 − e^(−k n / m))^k for n items added, rounded to 4 places.
- `best_k(m, n)` (a plain function): the k that minimises the false rate, round(m / n × ln 2), at least 1.

The checks add 1,000 words and measure the false-positive rate on 10,000 others against the formula.

In [ ]:
import hashlib, math

class BloomFilter:
    def __init__(self, m, k):
        ___

def best_k(m, n):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import math, random\ndef test_bloom():\n    \"probably present\"\n    b = BloomFilter(1000, 3)\n    ps = b.positions(\"apple\")\n    assert len(ps) == 3 and all(0 <= p < 1000 for p in ps) and ps == BloomFilter(1000, 3).positions(\"apple\")\n    assert \"apple\" not in b and len(b) == 0\n    b.add(\"apple\")\n    assert \"apple\" in b and len(b) == 1 and bin(b.bits).count(\"1\") <= 3\n    assert best_k(10_000, 1000) == 7 and best_k(10, 100) == 1\n    rng = random.Random(1)\n    words = {\"\".join(rng.choice(\"abcdefghij\") for _ in range(8)) for _ in range(1000)}\n    others = {\"\".join(rng.choice(\"klmnopqrst\") for _ in range(8)) for _ in range(10_000)}\n    bloom = BloomFilter(10_000, best_k(10_000, len(words)))\n    for w in words:\n        bloom.add(w)\n    assert all(w in bloom for w in words), \"no false negatives\"\n    rate = sum(1 for w in others if w in bloom) / len(others)\n    expected = bloom.expected_false_rate()\n    assert abs(expected - 0.0082) < 0.001 and rate < 3 * expected + 0.002, (rate, expected)")

## Warm-up

Two generators with the same **seed** produce the same "random" numbers. Computers' random numbers are **pseudo-random**: a deterministic sequence that looks random. That's a gift for testing: fix the seed, and a randomised program behaves the same every run.

## Randomness you can repeat

Python's `random` module uses the Mersenne Twister (Makoto Matsumoto and Takuji Nishimura, 1997). Use a `random.Random(seed)` **object** rather than the module's shared functions, and pass it into your functions: then callers and tests control the randomness, and nothing else in the program disturbs it. The main methods: `randint(a, b)` (both ends included), `randrange(n)` (0 to n − 1), `random()` (a float in [0, 1)), `choice`, `sample`, `shuffle`.

The Mersenne Twister is fine for simulations and algorithms, but predictable: never use it for passwords, tokens or anything secret. For those, the `secrets` module draws on the operating system's secure source.

Two kinds of randomised algorithm:

- **Las Vegas**: the answer is always correct; only the running time is random (quickselect with a random pivot).
- **Monte Carlo**: the running time is fixed; the answer is correct with high probability (Miller-Rabin, Freivalds' check). Repeating it shrinks the chance of error exponentially.

## Shuffling fairly

The **Fisher-Yates shuffle** (Ronald Fisher and Frank Yates described it for paper and pencil in 1938; Richard Durstenfeld gave the computer version in 1964): for i from the last position down to 1, swap item i with a **uniformly random** item from positions 0 to i (including i itself). Every one of the n! orders is equally likely, in O(n).

Small mistakes break it quietly: choosing from 0 to i − 1 (never leaving an item in place) gives Sattolo's algorithm, which only produces single cycles; choosing from the **whole** list each time gives nⁿ equally likely outcomes, which can't divide evenly among n! orders, so some orders come up more often.

Fisher-Yates gives each of the six orders about 10,000 times; the naive shuffle has 27 equally likely runs shared unevenly among 6 orders.

## A random pivot

Week 19's quicksort is O(n²) when the pivot is always the worst element. **Quickselect** finds the k-th smallest item without sorting: partition around a pivot, then continue only in the side that contains position k. With a **random** pivot, the expected work is O(n) on **every** input (each step throws away a good fraction on average), so no input can be crafted to make it slow every time. It's a Las Vegas algorithm, due to Tony Hoare (1961).

## Sampling a stream

You're reading a stream too long to store (a log file, network packets) and want k items chosen **uniformly at random** from all of them, without knowing the length in advance. **Reservoir sampling** (described by Alan Waterman; analysed by Jeffrey Vitter in 1985 as Algorithm R): keep the first k items; for the i-th item after that (counting from 0), pick a random j from 0 to i, and if j < k, replace reservoir slot j with it. Item i ends up in the sample with probability k / (i + 1) at that moment, and the later replacements keep every item's final chance equal to k / n.

## Monte Carlo

**Miller-Rabin** (Gary Miller, 1976; Michael Rabin, 1980) tests whether n is prime. Write n − 1 = 2ˢ × d with d odd. For a random base a, compute x = a^d mod n; if x is 1 or n − 1, a says "maybe prime". Otherwise square x up to s − 1 times; if it ever becomes n − 1, "maybe prime"; if not, n is **certainly composite** (a is a witness). For composite n at least three quarters of bases are witnesses, so 40 random rounds leave an error chance below 4⁻⁴⁰. (Fermat's test from Tuesday's theorem alone is fooled by Carmichael numbers like 561; Miller-Rabin isn't.) In fact, using the first 12 primes as bases is a proof of primality for every n below 3.3 × 10²⁴.

**Freivalds' check** (Rūsiņš Freivalds, 1977): to check whether A × B = C for n × n matrices without the O(n³) multiplication, pick a random 0/1 vector r and compare A(Br) with Cr: O(n²). If A × B ≠ C, each try catches it with probability at least 1/2.

## Practice

## Quiz

## Daily challenge

## Before you go

Shuffle a pack of cards with Fisher-Yates by hand for the first few positions (roll a die or use your phone for the random numbers). Tomorrow is the Forge: a toolkit of bits, number theory and randomness.